In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    roc_curve, roc_auc_score, auc
)
from sklearn.pipeline      import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel
import torch
from sklearn.model_selection import KFold
from sklearn.model_selection import ParameterGrid

#Custom data split function
#Input: imported data and embeddings from the data
def train_test_val_split(data, embeddings):
  test_index = data['test_edges']
  test_source = embeddings[test_index[:,0]]
  test_target = embeddings[test_index[:,1]]
  X_test = torch.cat([test_source, test_target], dim=1)
  y_test = data['test_labels']

  index = data['train_edges']
  source = embeddings[index[:, 0]]
  target = embeddings[index[:, 1]]
  X_train = torch.cat([source, target], dim=1)
  y_train = data['train_labels']

  val_index = data['val_edges']
  val_source = embeddings[val_index[:,0]]
  val_target = embeddings[val_index[:,1]]
  X_val = torch.cat([val_source, val_target], dim=1)
  y_val = data['val_labels']

  return X_train, X_test, y_train, y_test, X_val, y_val
#Output: training, testing, and validatoin sets

#Defining pipelines (models and scaling) and parameter grids (for tuning)
def define_models():
    rf_pipe = Pipeline([
        ("clf", RandomForestClassifier(random_state=137698, n_jobs=1))
    ])

    log_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=10000, random_state=137698))
    ])

    knn_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier())
    ])

    svc_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LinearSVC(max_iter=40000, random_state=137698))
    ])

    rf_grid = ParameterGrid({
        "clf__n_estimators": [100, 300, 500, 700, 1000],
        "clf__max_depth": [None, 10, 20, 30, 40, 50, 70, 100],
        "clf__min_samples_leaf": [1, 2, 3, 4, 5],
        "clf__min_samples_split": [2, 3, 5, 7, 10],
        "clf__max_features": ["sqrt", "log2", None],
        "clf__bootstrap": [True, False],
        "clf__class_weight": ['balanced']
    })

    log_grid = ParameterGrid({
        "clf__C": np.logspace(-6, 4, 15),
        "clf__penalty": ["l1", "l2"],
        "clf__solver": ["liblinear"],
        "clf__dual": [False],
        "clf__fit_intercept": [True, False],
        "clf__tol": [1e-5, 1e-4, 1e-3, 1e-2],
        "clf__class_weight": ["balanced"]
    })

    knn_grid = ParameterGrid({
        "clf__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21, 31],
        "clf__metric": ["euclidean", "manhattan", "chebyshev"],
        "clf__weights": ["uniform", "distance"],
        "clf__p": [1, 2],
        "clf__leaf_size": [10, 20, 30, 40, 50, 60, 80, 100],
        "clf__algorithm": ["auto", "ball_tree", "kd_tree", "brute"]
    })

    svc_grid = ParameterGrid([{
            "clf__C": np.logspace(-6, 4, 15),
            "clf__class_weight": ["balanced"],
            "clf__loss": ["squared_hinge"],
            "clf__penalty": ["l2"],
            "clf__dual": [False],
            "clf__fit_intercept": [True, False],
            "clf__tol": [1e-5, 1e-4, 1e-3, 1e-2]
        },{
            "clf__C": np.logspace(-6, 4, 15),
            "clf__class_weight": ["balanced"],
            "clf__loss": ["hinge"],
            "clf__penalty": ["l2"],
            "clf__dual": [True],
            "clf__fit_intercept": [True],
            "clf__tol": [1e-5, 1e-4, 1e-3, 1e-2]
        }])

    models = [
        ("Random Forest", rf_pipe, rf_grid),
        ("Logistic Regression", log_pipe, log_grid),
        ("KNN", knn_pipe, knn_grid),
        ("Linear SVC", svc_pipe, svc_grid),
    ]

    return models
#Output: list of tuples, each containing model name, pipeline, and parameter grid

#Selects best parameters for each model based on f1 macro scores
#Input: pipeline, grid, training and validation sets
def select_best_params(pipe, grid, X_train, y_train, X_val, y_val):
    best_f1 = -1
    best_params = None

    #sifts through all possible parameter combinations
    for params in grid:
        pipe.set_params(**params)
        pipe.fit(X_train, y_train)

        val_preds = pipe.predict(X_val)
        f1 = f1_score(y_val, val_preds, average='macro')

        if f1 > best_f1:
          best_f1 = f1
          best_params = params

    return best_params, best_f1
#Output: returns best parameter combination and its f1 score

#Overall tuning function implementing the smaller functions above
#Input: name of signed model
def tune(s_model):
  print(f"\n{s_model}")
  data = torch.load(s_model+'.pt')
  embeddings = data['embeddings']
  X_train, X_test, y_train, y_test, X_val, y_val = train_test_val_split(data, embeddings)
  models = define_models()

  for name, pipe, grid in models:
    print(f"Tuning {name}...")
    best_params, best_f1 = select_best_params(pipe, grid, X_train, y_train, X_val, y_val)
    print(f"Best Params: {best_params}")
    print(f"Val F1 (macro): {best_f1:.4f}")

    #Testing the optimized model across three different seeds and averaging the f1 scores
    seeds = [137698, 123, 42]
    results= []

    for seed in seeds:
      if "clf__random_state" in pipe.get_params():
        pipe.set_params(clf__random_state=seed)
      pipe.set_params(**best_params)
      pipe.fit(X_train, y_train)
      y_pred = pipe.predict(X_val)
      f1 = f1_score(y_val, y_pred)
      results.append(f1)

    average_f1 = np.mean(results)
    print(f"Avg F1: {average_f1}")

#Loop to tune all the models
#signedModels = ['sgcn', 'sdgnn', 'snea', 'sigat']
#for s_model in signedModels:
 # tune(s_model)